# Run cells from AllenDB

In [1]:
import os, sys, json
import matplotlib.pyplot as plt
import numpy as np

# !pip install --quiet allensdk neuron > /dev/null 2>&1

# Import single_cells modules
from modules import download_cell

# Download Cell

### Set Parameters

In [4]:
cell_name = 'PV'
cell_dir = f"/home/hrbncv/mods/ACT/data/{cell_name}"    #f"cells/{cell_name}"
if os.path.isdir(cell_dir): os.chdir(cell_dir), print(f'Download Loc: {cell_dir}')

# http://celltypes.brain-map.org/mouse/experiment/electrophysiology/'specimen_id'
specimen_id = 484635029
model_type = 'perisomatic' #or 'all active'
tunes_dir = 'orig'
model_dir = '' #f'OriginalFromAllenDB/{specimen_id}_{model_type}'

Download Loc: /home/hrbncv/mods/ACT/data/PV


### Download Cell (if not already downloaded)

In [5]:
# (Optional) list available bundles for a specimen
download_cell.list_ADB_models(specimen_id)                             # all
download_cell.list_ADB_models(specimen_id, filter_type=model_type)  # filtered

# Download a bundle
cell_info = download_cell.download_ADB_cell(
    specimen_id=specimen_id,
    model_type=model_type,          # or "all active"
    tunes_dir=tunes_dir,    # base dir
    cache_stimulus=False,              # skip big NWB
    subdir=model_dir,          # None ='tunes_dir/', else ='tunes_dir'/'model_dir/'
    match="contains",                  # name matching behavior
    quiet=False,
)
cell_info["model_id"], cell_info["model_name"], cell_info["tunes_dir"]

# List the files you just pulled
len(cell_info["files"]), cell_info["files"][:5]

Models for specimen_id=484635029:
  485602029  Biophysical - perisomatic_Pvalb-IRES-Cre;Ai14-201791.05.01.01
  496538965  Biophysical - all active_Pvalb-IRES-Cre;Ai14-201791.05.01.01
Models for specimen_id=484635029:
  485602029  Biophysical - perisomatic_Pvalb-IRES-Cre;Ai14-201791.05.01.01


2025-09-15 21:14:10,534 allensdk.api.api.retrieve_file_over_http INFO     Downloading URL: http://api.brain-map.org/api/v2/well_known_file_download/496079601
2025-09-15 21:14:10,783 allensdk.api.api.retrieve_file_over_http INFO     Downloading URL: http://api.brain-map.org/api/v2/well_known_file_download/496605128
2025-09-15 21:14:10,961 allensdk.api.api.retrieve_file_over_http INFO     Downloading URL: http://api.brain-map.org/api/v2/well_known_file_download/395337293
2025-09-15 21:14:11,145 allensdk.api.api.retrieve_file_over_http INFO     Downloading URL: http://api.brain-map.org/api/v2/well_known_file_download/395337054
2025-09-15 21:14:11,471 allensdk.api.api.retrieve_file_over_http INFO     Downloading URL: http://api.brain-map.org/api/v2/well_known_file_download/395337225
2025-09-15 21:14:11,654 allensdk.api.api.retrieve_file_over_http INFO     Downloading URL: http://api.brain-map.org/api/v2/well_known_file_download/395337019
2025-09-15 21:14:11,835 allensdk.api.api.retrieve_fi

Downloaded model_id=485602029 (Biophysical - perisomatic_Pvalb-IRES-Cre;Ai14-201791.05.01.01) for specimen_id=484635029


(20,
 ['orig/484635029_fit.json',
  'orig/Pvalb-IRES-Cre_Ai14-201791.05.01.01_496079599_m.swc',
  'orig/Pvalb-IRES-Cre_Ai14-201791.05.01.01_496079599_marker_m.swc',
  'orig/manifest.json',
  'orig/modfiles/CaDynamics.mod'])

### Compile Modfiles

In [4]:
os.chdir(f'{tunes_dir}/{model_dir}')

# update_modfiles = None # or path/link to modfiles TODO
# if update_modfiles is not None:
#     !git clone update_modfiles

# if already compiled then lets delete the folder and force a recompile
if os.path.isdir('modfiles/x86_64'):
    os.system("rm -rf modfiles/x86_64")
# compile the mod files
if not os.path.isdir("modfiles/x86_64"):
    # !nrnivmodl modfiles > /dev/null 2>&1
    os.chdir('modfiles')
    os.system("nrnivmodl")
    os.chdir("..")


from neuron import h
h.load_file("stdrun.hoc")  # Required to use h.run()
h.nrn_load_dll("x86_64/.libs/libnrnmech.so")

/home/hrbncv/mods/ACT/data/SST2/seg_tuned/modfiles
Mod files: "./CaDynamics.mod" "./Ca_HVA.mod" "./Ca_LVA.mod" "./Ih.mod" "./Im.mod" "./Im_v2.mod" "./Kd.mod" "./K_P.mod" "./K_T.mod" "./Kv2like.mod" "./Kv3_1.mod" "./Nap.mod" "./NaTa.mod" "./NaTs.mod" "./NaV.mod" "./SK.mod"

Creating 'x86_64' directory for .o files.

 -> Compiling mod_func.cpp
 -> NMODL ../CaDynamics.mod
 -> NMODL ../Ca_HVA.mod
 -> NMODL ../Ca_LVA.mod


/home/hrbncv/miniconda3/envs/BMTK/bin/nrnivmodl:10: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import working_set
Translating CaDynamics.mod into /home/hrbncv/mods/ACT/data/SST2/seg_tuned/modfiles/x86_64/CaDynamics.c
Thread Safe
Translating Ca_LVA.mod into /home/hrbncv/mods/ACT/data/SST2/seg_tuned/modfiles/x86_64/Ca_LVA.c
Translating Ca_HVA.mod into /home/hrbncv/mods/ACT/data/SST2/seg_tuned/modfiles/x86_64/Ca_HVA.c
Thread Safe
Thread Safe
Translating Ih.mod into /home/hrbncv/mods/ACT/data/SST2/seg_tuned/modfiles/x86_64/Ih.c
Thread Safe
Translating Im.mod into /home/hrbncv/mods/ACT/data/SST2/seg_tuned/modfiles/x86_64/Im.c
Thread Safe
Translating Im_v2.mod into /home/hrbncv/mods/ACT/data/SST2/seg_tuned/modfiles/x86_64/Im_v2.c
Translating Kd.mod into /home/hrbncv/mods/ACT/data/SST2/seg_tuned/modfiles/x86_64/Kd.c
Thread Safe
Thread Safe
Translating K_P.mod into /home/hrbncv/mods/ACT/data/SST2/s

 -> NMODL ../Ih.mod
 -> NMODL ../Im.mod
 -> NMODL ../Im_v2.mod
 -> NMODL ../Kd.mod
 -> NMODL ../K_P.mod
 -> NMODL ../K_T.mod
 -> NMODL ../Kv2like.mod
 -> NMODL ../Kv3_1.mod
 -> NMODL ../Nap.mod
 -> NMODL ../NaTa.mod
 -> NMODL ../NaTs.mod
 -> NMODL ../NaV.mod
 -> NMODL ../SK.mod
 -> Compiling CaDynamics.c
 -> Compiling Ca_HVA.c
 -> Compiling Ca_LVA.c
 -> Compiling Ih.c
 -> Compiling Im.c
 -> Compiling Im_v2.c
 -> Compiling Kd.c
 -> Compiling K_P.c
 -> Compiling K_T.c
 -> Compiling Kv2like.c
 -> Compiling Kv3_1.c
 -> Compiling Nap.c
 -> Compiling NaTa.c
 -> Compiling NaTs.c
 -> Compiling NaV.c
 -> Compiling SK.c
 => LINKING shared library ./libnrnmech.so
 => LINKING executable ./special LDFLAGS are:    -pthread
Successfully created x86_64/special


--No graphics will be displayed.


0.0